# Date sources
Compares date sources (Wikidata property vs the Wikidata description, read by enrichment step 02), their precision, and the age of citizenship polities. Dates estimated from life expectancy are derived, not a source, and are left out.

In [1]:
import tempfile
import numpy as np
import polars as pl
import duckdb
from style import *

SEED = 42
np.random.seed(SEED)

DB_PATH = '../data/cultura/humans_clean_enriched_v3.duckdb'
SUB_YEAR = ('day', 'month')
YEAR_OR_FINER = ('day', 'month', 'year')

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

apply_style()

## Connect

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
con.sql(f"SET memory_limit='4GB'; SET threads=6; SET temp_directory='{tempfile.mkdtemp()}';")
con.sql("CREATE TEMP MACRO rule_of(d) AS coalesce(d.field_provenance['year'].rule, '')")
con.sql("CREATE TEMP MACRO from_description(d) AS rule_of(d) ILIKE '%description%'")
con.sql("CREATE TEMP MACRO from_wikidata(d) AS NOT from_description(d) AND rule_of(d) NOT ILIKE '%life expectancy%'")
con.sql("CREATE TEMP MACRO wikidata_date(dates, levels) AS coalesce(list_bool_or(list_transform(dates, d -> from_wikidata(d) AND (levels IS NULL OR list_contains(levels, d.precision)))), false)")
con.sql("CREATE TEMP MACRO description_date(dates, levels) AS coalesce(list_bool_or(list_transform(dates, d -> from_description(d) AND (levels IS NULL OR list_contains(levels, d.precision)))), false)")

## Load data
### Date-source flags

In [3]:
precision_levels = {'sub_year': list(SUB_YEAR), 'year_or_finer': list(YEAR_OR_FINER)}

con.sql("""
    CREATE OR REPLACE TEMP TABLE dated_place AS
    SELECT
        entity.qid                              AS qid,
        existence.inception.year IS NOT NULL    AS has_inception,
        existence.dissolution.year IS NOT NULL  AS has_dissolution
    FROM place
""")

con.sql("""
    CREATE OR REPLACE TEMP TABLE individual_citizenship AS
    SELECT
        entity.qid                                 AS qid,
        unnest(country_of_citizenship).entity.qid  AS place_qid
    FROM individual
""")

con.sql("""
    CREATE OR REPLACE TEMP TABLE citizenship_span AS
    SELECT
        individual_citizenship.qid,
        bool_or(dated_place.has_inception)    AS cit_inception,
        bool_or(dated_place.has_dissolution)  AS cit_dissolution
    FROM individual_citizenship
    JOIN dated_place
      ON dated_place.qid = individual_citizenship.place_qid
    GROUP BY individual_citizenship.qid
""")

con.sql("""
    CREATE OR REPLACE TEMP TABLE work_date_flags AS
    SELECT
        individual_work.qid,
        bool_or(work.publication_date.year IS NOT NULL)                             AS pub,
        bool_or(list_contains($sub_year, work.publication_date.precision))          AS pub_sub,
        bool_or(list_contains($year_or_finer, work.publication_date.precision))     AS pub_year,
        bool_or(work.inception.year IS NOT NULL)                                    AS winc,
        bool_or(list_contains($sub_year, work.inception.precision))                 AS winc_sub,
        bool_or(list_contains($year_or_finer, work.inception.precision))            AS winc_year
    FROM individual_work
    JOIN work
      ON work.entity.qid = individual_work.work_qid
    GROUP BY individual_work.qid
""", params=precision_levels)

con.sql("""
    CREATE OR REPLACE TEMP TABLE individual_date_flags AS
    SELECT
        wikidata_date(individual.birth_date, NULL)                                                  AS birth_wd,
        wikidata_date(individual.birth_date, $sub_year)        AS birth_wd_sub,
        wikidata_date(individual.birth_date, $year_or_finer)   AS birth_wd_year,
        description_date(individual.birth_date, NULL)                                                       AS birth_desc,
        description_date(individual.birth_date, $sub_year)             AS birth_desc_sub,
        description_date(individual.birth_date, $year_or_finer)        AS birth_desc_year,
        wikidata_date(individual.death_date, NULL)                                                  AS death_wd,
        wikidata_date(individual.death_date, $sub_year)        AS death_wd_sub,
        wikidata_date(individual.death_date, $year_or_finer)   AS death_wd_year,
        description_date(individual.death_date, NULL)                                                       AS death_desc,
        description_date(individual.death_date, $sub_year)             AS death_desc_sub,
        description_date(individual.death_date, $year_or_finer)        AS death_desc_year,
        wikidata_date(individual.floruit_date, NULL)                                                AS floruit_wd,
        wikidata_date(individual.floruit_date, $sub_year)    AS floruit_wd_sub,
        wikidata_date(individual.floruit_date, $year_or_finer) AS floruit_wd_year,
        description_date(individual.floruit_date, NULL)                                                     AS floruit_desc,
        description_date(individual.floruit_date, $sub_year)         AS floruit_desc_sub,
        description_date(individual.floruit_date, $year_or_finer)    AS floruit_desc_year,
        individual.works_period.first_year IS NOT NULL   AS has_works_period,
        coalesce(citizenship_span.cit_inception, false)        AS cit_inception,
        coalesce(citizenship_span.cit_dissolution, false)      AS cit_dissolution,
        coalesce(birth_place.has_inception, false)       AS bp_inception,
        coalesce(birth_place.has_dissolution, false)     AS bp_dissolution,
        coalesce(death_place.has_inception, false)       AS dp_inception,
        coalesce(death_place.has_dissolution, false)     AS dp_dissolution,
        coalesce(work_date_flags.pub, false)                  AS pub,
        coalesce(work_date_flags.pub_sub, false)              AS pub_sub,
        coalesce(work_date_flags.pub_year, false)             AS pub_year,
        coalesce(work_date_flags.winc, false)                 AS winc,
        coalesce(work_date_flags.winc_sub, false)             AS winc_sub,
        coalesce(work_date_flags.winc_year, false)            AS winc_year
    FROM individual
    LEFT JOIN citizenship_span
           ON citizenship_span.qid = individual.entity.qid
    LEFT JOIN dated_place AS birth_place
           ON birth_place.qid = individual.place_of_birth.entity.qid
    LEFT JOIN dated_place AS death_place
           ON death_place.qid = individual.place_of_death.entity.qid
    LEFT JOIN work_date_flags
           ON work_date_flags.qid = individual.entity.qid
""", params=precision_levels)

flag_combos = con.sql("""
    SELECT
        *,
        count(*) AS n
    FROM individual_date_flags
    GROUP BY ALL
""").pl()

TOTAL_INDIVIDUALS = con.sql('SELECT count(*) FROM individual').fetchone()[0]
assert flag_combos['n'].sum() == TOTAL_INDIVIDUALS
print(f'individuals: {TOTAL_INDIVIDUALS:,}  |  flag combinations: {flag_combos.height:,}')
flag_combos.sort('n', descending=True).head()

individuals: 13,002,897  |  flag combinations: 8,505


birth_wd,birth_wd_sub,birth_wd_year,birth_desc,birth_desc_sub,birth_desc_year,death_wd,death_wd_sub,death_wd_year,death_desc,death_desc_sub,death_desc_year,floruit_wd,floruit_wd_sub,floruit_wd_year,floruit_desc,floruit_desc_sub,floruit_desc_year,has_works_period,cit_inception,cit_dissolution,bp_inception,bp_dissolution,dp_inception,dp_dissolution,pub,pub_sub,pub_year,winc,winc_sub,winc_year,n
bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64
false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,1989520
false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,true,true,true,false,false,false,1928784
true,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,916570
true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,838550
false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,591778


In [4]:
con.close()

## Date-source coverage

In [5]:
SPAN = '—'

def n_where(expr):
    return flag_combos.filter(expr)['n'].sum()

def point_row(label, flag):
    n, n_sub = n_where(pl.col(flag)), n_where(pl.col(f'{flag}_sub'))
    return label, n, f'{n_sub / n * 100:.2f}%' if n else SPAN

def span_row(label, flag):
    return label, n_where(pl.col(flag)), SPAN

sources = [
    point_row('Birth date — Wikidata property', 'birth_wd'),
    point_row('Death date — Wikidata property', 'death_wd'),
    point_row('Floruit date — Wikidata property', 'floruit_wd'),
    point_row('Birth date — Wikidata description', 'birth_desc'),
    point_row('Death date — Wikidata description', 'death_desc'),
    point_row('Floruit date — Wikidata description', 'floruit_desc'),
    span_row('Country of citizenship — dissolution', 'cit_dissolution'),
    span_row('Country of citizenship — inception', 'cit_inception'),
    span_row('Birthplace — inception', 'bp_inception'),
    span_row('Birthplace — dissolution', 'bp_dissolution'),
    span_row('Deathplace — inception', 'dp_inception'),
    span_row('Deathplace — dissolution', 'dp_dissolution'),
    point_row('Works — publication date', 'pub'),
    point_row('Works — inception date', 'winc'),
]

point_flags = [f'{k}_{s}' for k in ['birth', 'death', 'floruit'] for s in ['wd', 'desc']]
any_date = pl.any_horizontal(point_flags + ['cit_inception', 'cit_dissolution', 'bp_inception',
                                            'bp_dissolution', 'dp_inception', 'dp_dissolution', 'pub', 'winc'])
year_precise = pl.any_horizontal([f'{f}_year' for f in point_flags] + ['pub_year', 'winc_year'])
n_any_date, n_year_precise = n_where(any_date), n_where(year_precise)

date_source_coverage = pl.DataFrame(
    [{'source': s, 'pct_individuals': f'{n / TOTAL_INDIVIDUALS * 100:.2f}%', 'precision_note': note}
     for s, n, note in sources]
    + [{'source': 'TOTAL — at least one date',
        'pct_individuals': f'{n_any_date / TOTAL_INDIVIDUALS * 100:.2f}%',
        'precision_note': f'{n_year_precise / n_any_date * 100:.2f}% (year or finer)'}]
)
print(f'Total unique individuals: {TOTAL_INDIVIDUALS:,}')
date_source_coverage

Total unique individuals: 13,002,897


source,pct_individuals,precision_note
str,str,str
"""Birth date — Wikidata property""","""57.59%""","""62.97%"""
"""Death date — Wikidata property""","""27.16%""","""70.39%"""
"""Floruit date — Wikidata property""","""0.53%""","""2.04%"""
"""Birth date — Wikidata description""","""15.75%""","""0.00%"""
"""Death date — Wikidata description""","""11.47%""","""0.00%"""
"""Floruit date — Wikidata description""","""0.17%""","""0.00%"""
"""Country of citizenship — dissolution""","""7.63%""","""—"""
"""Country of citizenship — inception""","""42.72%""","""—"""
"""Birthplace — inception""","""14.92%""","""—"""


## Date-source cascade

In [6]:
ENRICHMENT_STEPS = [
    ('Birth/death/floruit date from a Wikidata property', pl.any_horizontal('birth_wd', 'death_wd', 'floruit_wd')),
    ('Birth/death/floruit year from the Wikidata description', pl.any_horizontal('birth_desc', 'death_desc', 'floruit_desc')),
    ('Works min/max dates from Wikidata', pl.col('has_works_period')),
    ('Country-of-citizenship span from Wikidata', pl.any_horizontal('cit_inception', 'cit_dissolution')),
    ('Birth/death place span from Wikidata', pl.any_horizontal('bp_inception', 'bp_dissolution', 'dp_inception', 'dp_dissolution')),
]

covered = pl.lit(False)
cascade_rows = []
for label, flag in ENRICHMENT_STEPS:
    added = n_where(flag & ~covered)
    covered = covered | flag
    cascade_rows.append({'step': label, 'added': added,
                         'added_pct': added / TOTAL_INDIVIDUALS * 100,
                         'cum_pct': n_where(covered) / TOTAL_INDIVIDUALS * 100})

cascade = pl.DataFrame(cascade_rows)
assert cascade['added'].sum() == n_where(covered)
cascade

step,added,added_pct,cum_pct
str,i64,f64,f64
"""Birth/death/floruit date from a Wikidata property""",7794340,59.943103,59.943103
"""Birth/death/floruit year from the Wikidata description""",169563,1.30404,61.247144
"""Works min/max dates from Wikidata""",2168760,16.679052,77.926196
"""Country-of-citizenship span from Wikidata""",862837,6.635729,84.561925
"""Birth/death place span from Wikidata""",17877,0.137485,84.69941


## Paper table

### Sources of date information

In [7]:
def fmt_pct(p):
    return '<0.1%' if 0 < p < 0.1 else f'{p:.2f}%'

print('| Method | Unique individuals dated | Cumulative coverage |')
print('|---|---:|---:|')
for row in cascade.iter_rows(named=True):
    print(f"| {row['step']} | +{fmt_pct(row['added_pct'])} | {row['cum_pct']:.2f}% |")
final_coverage = cascade[-1, 'cum_pct']
print(f'| **Total with date/period** | **{final_coverage:.2f}%** | **{final_coverage:.2f}%** |')

| Method | Unique individuals dated | Cumulative coverage |
|---|---:|---:|
| Birth/death/floruit date from a Wikidata property | +59.94% | 59.94% |
| Birth/death/floruit year from the Wikidata description | +1.30% | 61.25% |
| Works min/max dates from Wikidata | +16.68% | 77.93% |
| Country-of-citizenship span from Wikidata | +6.64% | 84.56% |
| Birth/death place span from Wikidata | +0.14% | 84.70% |
| **Total with date/period** | **84.70%** | **84.70%** |
